In [1]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from datasets import Dataset

/Users/mthariqsultand/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
df = pd.read_csv('bersatu_finetune.csv')

# pilih kolom
df = df[['clean_textdisplay', 'label']].copy()

# hapus data kosong
df = df.dropna()

# rapihin label
df['label'] = (
    df['label']
    .astype(str)
    .str.lower()
    .str.strip()
)


In [3]:
# mapping label indonesia
label_mapping = {
    'negatif': 0,
    'netral': 1,
    'positif': 2
}

df['label'] = df['label'].map(label_mapping)

# cek hasil
print(df['label'].value_counts())
print(df.head())

label
1    800
0    800
2    800
Name: count, dtype: int64
                                   clean_textdisplay  label
0  ada ga ada mobil listrik pabrikan nikel terus ...      1
1  akhirnya persaingan negara china dengan electr...      1
2  baterai electric vehicle di recycle menjadi bl...      1
3  bcara baterai mobil bensin solar pun ada bater...      1
4  betul dokter mobil yg ceritakan ku pake mobil ...      1


In [4]:
df = df.dropna(subset=['label'])

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

# Convert to HuggingFace Dataset
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [5]:
MODEL_NAME = "indobenchmark/indobert-base-p1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [6]:
def tokenize_function(example):
    return tokenizer(
        example['clean_textdisplay'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/1920 [00:00<?, ? examples/s]

Map:   0%|          | 0/480 [00:00<?, ? examples/s]

In [7]:

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc
    }

In [9]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_dir="./logs",
    logging_steps=10
)

In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [11]:
trainer.train()

/Users/mthariqsultand/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.946100,0.906803,0.612500
2,0.743900,0.840224,0.620833


/Users/mthariqsultand/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=120, training_loss=0.9118452747662862, metrics={'train_runtime': 122.536, 'train_samples_per_second': 31.338, 'train_steps_per_second': 0.979, 'total_flos': 252588881018880.0, 'train_loss': 0.9118452747662862, 'epoch': 2.0})

In [12]:
predictions = trainer.predict(test_dataset)

y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

print("\nAccuracy:")
print(accuracy_score(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

/Users/mthariqsultand/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)



Accuracy:
0.6208333333333333

Classification Report:
              precision    recall  f1-score   support

           0       0.58      0.54      0.56       160
           1       0.71      0.59      0.64       160
           2       0.59      0.73      0.65       160

    accuracy                           0.62       480
   macro avg       0.63      0.62      0.62       480
weighted avg       0.63      0.62      0.62       480


Confusion Matrix:
[[ 87  25  48]
 [ 33  94  33]
 [ 29  14 117]]


In [13]:
df['clean_textdisplay'].duplicated().sum()

np.int64(42)